In [2]:
import coiled

import fsspec
import s3fs
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging
from flox.xarray import xarray_reduce
import numpy as np
import pytz
import dask
import re
import sparse
import time
from datetime import datetime
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy

import pygwalker as pyg

# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

In [3]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [4]:
fs = fsspec.filesystem("s3", requester_pays=True)

In [36]:
cluster = coiled.Cluster(
    name="LULUCF_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=10,
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r5.2xlarge", # memory optimized AWS EC2 instances
    worker_vm_types="r5.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                ╷                                                             │
│   Package      │ Note                                                        │
│ ╶──────────────┼───────────────────────────────────────────────────────────╴ │
│   flox         │ Wheel built from ~/flox-0.10.3.tar.gz                       │
│                ╵                                                             │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│             ╷                                                    ╷           │
│   Package   │ Error                                              │ Level     │
│ ╶───────────┼────────────────────────────────────────────────────┼─────────╴ │
│   pygwalker │ Pip check had the following issues that need       │ Warning   │
│             │ resolving:                                         │           │
│             │ pygwalker 0.3.17 has requirement                   │           │
│             │ segment-analytics-python==2.2.3, but you have      │           │
│             │ segment-analytics-python 2.2.2.                    │           │
│             ╵                                                    ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

In [ ]:
client.restart() 

In [ ]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

In [ ]:
local_client.shutdown()

In [6]:
def timestr():

    # Define the Eastern Time timezone
    eastern = pytz.timezone('US/Eastern')

    # Get the current time in UTC and convert to Eastern Time
    eastern_time = datetime.now(eastern)

    # Format the time as a string
    return eastern_time.strftime("%Y%m%d_%H_%M_%S")

In [7]:
# per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682201ec-1f84-800a-a9f9-c9564f613208
def list_folder_uris(base_uri):

    # Initializes S3 filesystem
    fs = s3fs.S3FileSystem(anon=False)  # Set anon=True if public bucket
    
    # Lists all files in the directory
    all_files = fs.ls(base_uri)
    
    # Filters for GeoTIFFs
    tif_files = [f"s3://{f}" for f in all_files if f.endswith(".tif")]
    
    # Converts to a Pandas Series
    series = pd.Series(tif_files)
    
    return series

In [18]:
# Node codes output from model. Covers entire decision tree. Make sure that node codes are right-padded with 0s to 7 digits! 
# Otherwise, only the node codes that are seven digits without 0s will be matched with the node code rasters and output. 
# TODO: I may have accidentally missed some node codes when copying them from the decision tree. Check!
node_codes = np.array([
    1110000, 1120000, 1210000, 1220000, 2111000, 2112000,
    2121100, 2121200, 2122100, 2122200, 2123100, 2123200,
    2124100, 2124200, 2125100, 2125200, 2211100, 2211200, 2212110, 2212120, 
    2212210, 2212220, 2213110, 2213120, 2213210, 2213220,
    2214100, 2214200, 2215100, 2215200, 2221100, 2221200, 
    2222100, 2222200, 2223100, 2223200, 3110000, 3120000, 
    3211111, 3211112, 3211121, 3211122, 3211211, 3211212,
    3211221, 3211222, 3212111, 3212112, 3212121, 3212122,
    3212211, 3212212, 3212221, 3212222, 3221110, 3221120,
    3221210, 3221220, 3222111, 3222112, 3222121, 3222122,
    3222210, 3222220, 4100000, 4210000, 4220000, 4310000,
    4320000, 5100000, 5210000, 5220000, 5310000, 5320000],
dtype=np.uint32)

# # Node codes output from model for 2x2 test area in DRC (23_-5_25_-3)
# node_codes = np.array([3222111, 2212120, 3222121, 3212122, 2212110, 3212121, 3222210, 2223200, 
#                        2221200, 3212222, 2221100, 5220000, 4100000, 2223100,
#                        2212220, 3212221, 2211200, 5210000, 2211100, 4220000, 2212210, 2214100, 2214200, 4210000, 2215200], 
#                       dtype=np.uint32)

In [17]:
gadm_adm0_ids = np.array([  0.,   4.,   8.,  10.,  12.,  16.,  20.,  24.,  28.,  31.,  32.,
        36.,  40.,  44.,  48.,  50.,  51.,  52.,  56.,  60.,  64.,  68.,
        70.,  72.,  74.,  76.,  84.,  86.,  90.,  92.,  96., 100., 104.,
       108., 112., 116., 120., 124., 132., 136., 140., 144., 148., 152.,
       156., 158., 162., 166., 170., 174., 175., 178., 180., 184., 188.,
       191., 192., 196., 203., 204., 208., 212., 214., 218., 222., 226.,
       231., 232., 233., 234., 238., 239., 242., 246., 248., 250., 254.,
       258., 260., 262., 266., 268., 270., 275., 276., 288., 292., 296.,
       300., 304., 308., 312., 316., 320., 324., 328., 332., 334., 336.,
       340., 348., 352., 356., 360., 364., 368., 372., 376., 380., 384.,
       388., 392., 398., 400., 404., 408., 410., 414., 417., 418., 422.,
       426., 428., 430., 434., 438., 440., 442., 450., 454., 458., 462.,
       466., 470., 474., 478., 480., 484., 492., 496., 498., 499., 500.,
       504., 508., 512., 516., 520., 524., 528., 531., 533., 534., 535.,
       540., 548., 554., 558., 562., 566.,70., 574., 578., 580., 581.,
       583., 584., 585., 586., 591., 598., 600., 604., 608., 612., 616.,
       620., 624., 626., 630., 634., 638., 642., 643., 646., 652., 654.,
       659., 660., 662., 663., 666., 670., 674., 678., 682., 686., 688.,
       690., 694., 702., 703., 704., 705., 706., 710., 716., 724., 728.,
       729., 732., 740., 744., 748., 752., 756., 760., 762., 764., 768.,
       772., 776., 780., 784., 788., 792., 795., 796., 798., 800., 804.,
       807., 818., 826., 831., 832., 833., 834., 840., 850., 854., 858.,
       860., 862., 876., 882., 887., 894.], dtype=np.uint16)

In [8]:
# Extracts some metadata/chunk properties to add to the output dataframe
def parse_metadata_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_pixel_yr_\d{4}_\d{4}\.tif$"
    match = re.search(pattern, uri)

    if match:
        return match.group(1)
    else:
        return None

In [9]:
def make_xarray_chunks(tile_uris, chunk_size):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True,
        chunks={'x': chunk_size, 'y':chunk_size}
    ).squeeze().persist()

    return xarray_chunks

In [10]:
def align_with_nodes(analysis_layer, nodes):
    analysis_layer_sub, nodes_aligned = xr.align(analysis_layer, nodes, join="inner")
    return analysis_layer_sub, nodes_aligned

In [11]:
def xarray_reduction_sum_count(analysis_layer, node_data):

    reductions = {}

    for func in ["sum", "count"]:
        reduced = xarray_reduce(
            analysis_layer.band_data,
            node_data,
            func=func,
            keep_attrs=True,
            expected_groups=(node_codes),
            reindex=ReindexStrategy(
                blockwise=False,
                array_type=ReindexArrayType.SPARSE_COO
            ),
            fill_value=0
        )

        # Rename variables to reflect reduction type
        if isinstance(reduced, xr.Dataset):
            renamed = reduced.rename({var: f"{var}_{func}" for var in reduced.data_vars})
        else:  # it's a DataArray
            renamed = reduced.rename(f"{reduced.name}_{func}")

        reductions[func] = renamed

    # Merge results: handle Dataset or DataArray combinations
    result = xr.merge([r if isinstance(r, xr.Dataset) else r.to_dataset() for r in reductions.values()])

    return result

In [12]:
def xarray_reduction(flux_cube, nodes_aligned_data, adm0_data):

    data_cube_by_node = xarray_reduce(
        flux_cube,
        nodes_aligned_data,
        adm0_data,
        func='sum',
        keep_attrs=True,
        expected_groups=(node_codes),
        reindex=ReindexStrategy(
            blockwise=False, array_type=ReindexArrayType.SPARSE_COO
        ),
        fill_value=0   
    )

    return data_cube_by_node

In [113]:
# Converts flox output to dataframe and does some processing of it
def create_interval_df(coord_dict):

    df = pd.DataFrame(coord_dict)

    # Replaces numeric values for outputs with names
    df['flux_type'] = df['flux_type'].replace({0: gross_emis_CO2_output_pattern, 1: gross_emis_all_gases_output_pattern, 
                                               2: gross_remv_all_pools_output_pattern, 3: net_flux_output_pattern, 4: "area__ha"})
    
    # Classifies the node_codes by larger groupings
    df['node_grp'] = df['state_node'].apply(classify_node)

    # Makes node_codes into strings
    df['state_node'] = 'n' + df['state_node'].astype(str)

    # Adds the interval end year to the dataframe
    df['interval_end'] = interval_end_year
    
    # print(df)

    return df

In [114]:
# Calculates flux densities (Mg CO2 or CO2e/ha)
# Per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682a8c76-0618-800a-a201-18fd404a281f
def calculate_interval_flux_densities(df):

    # Step 1: Filters out area and flux data
    area_df = df[df['flux_type'] == 'area__ha'].copy()
    flux_df = df[df['flux_type'] != 'area__ha'].copy()
    
    # Step 2: Merges flux data with area data on matching keys
    merged = pd.merge(
        flux_df,
        area_df[['state_node', 'gadm_adm0', 'interval_end', 'value']],
        on=['state_node', 'gadm_adm0', 'interval_end'],
        how='left',
        suffixes=('', '_area')
    )
    
    # Step 3: Computes per-hectare flux
    merged['value_per_ha'] = merged['value'] / merged['value_area']
    
    # Step 4: Prepares flux density rows to append
    new_rows = merged.copy()
    new_rows['flux_type'] = new_rows['flux_type'] + '__per_ha'
    new_rows['value'] = new_rows['value_per_ha']
    new_rows = new_rows.drop(columns=['value_area', 'value_per_ha'])
    
    # Step 5: Appends flux density rows to original dataframe
    result_df = pd.concat([df, new_rows], ignore_index=True)

In [14]:
# Reclassifies state nodes to broad categories
def classify_node(state_node):
    
    node_str = str(state_node)
    first_digit = int(node_str[0])
    # print(first_digit)

    # For broad classes that can be categorized using just the first digit
    one_digit_map = {
        1: 'forest_gain',
        2: 'forest_loss',
        4: 'cropland',
        5: 'grassland'
    }

    # For broad classes that need to be categorized using the first three digits
    three_digit_map = {
        321: 'disturbed_forest',
        322: 'stable_forest'
        # Add more as needed
    }
    
    if first_digit == 3:
        prefix = int(node_str[:3])
        # print(prefix)
        # print(two_digit_map.get(prefix, 'unknown_3x'))
        return three_digit_map.get(prefix, 'unknown_3x')
    else:
        return one_digit_map.get(first_digit, 'unknown')

Code to run zonal stats

In [37]:
# uri components

# model_version = "version_0_3_2"
# run_date = "20250507"
# chunk_size = 4000

model_version = "version_0_3_3"
run_date = "20250511"
chunk_size = 10000

output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"
# interval_end_years = [2016]
# interval_end_years = [2020]
# interval_end_years = [2016, 2017, 2018]
interval_end_years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

tile_id = '00N_020E'

# s3 folders for inputs
gross_emis_CO2_folder = f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
gross_emis_all_gases_folder = f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
gross_remv_all_pools_folder = f"{output_path}gross_removals__all_C_pools__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
net_flux_all_pools_CO2_folder = f"{output_path}net_flux__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/40000_pixels/{run_date}/"
node_folder = f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/40000_pixels/{run_date}/"

adm0_folder = "s3://gfw2-data/gadm_administrative_boundaries/v4.1/v4.1.64__from_gfw-data-lake/raster/epsg-4326/10/40000/adm0/gdal-geotiff/" #GADM v4.1
pixel_area_folder = "s3://gfw2-data/analyses/umd_area_2013__from_gfw-data-lake/v1.10/raster/epsg-4326/10/40000/area_m/gdal-geotiff/"

In [38]:
combined_df = pd.DataFrame()
analysis_start_time = time.time()

adm0_uris = pd.Series([f"s3://gfw2-data/gadm_administrative_boundaries/v4.1/v4.1.64__from_gfw-data-lake/raster/epsg-4326/10/40000/adm0/gdal-geotiff/{tile_id}.tif"])
pixel_area_uris = pd.Series([f"s3://gfw2-data/analyses/umd_area_2013__from_gfw-data-lake/v1.10/raster/epsg-4326/10/40000/area_m/gdal-geotiff/{tile_id}.tif"])

print(f"Reading inputs that apply to all intervals: {timestr()}")
print(f"   Reading adm0: {timestr()}")
adm0_xarray_chunks = make_xarray_chunks(adm0_uris, chunk_size)
print(f"   Reading pixel_area: {timestr()}")
pixel_area_xarray_chunks = make_xarray_chunks(pixel_area_uris, chunk_size)

# print("adm0_xarray_chunks:", adm0_xarray_chunks)
# print("pixel_area_xarray_chunks:", pixel_area_xarray_chunks)

for interval_end_year in interval_end_years:

    interval = f"{interval_end_year-1}_{interval_end_year}"
    # print(interval)

    print(f"Processing {interval}: {timestr()}")
    interval_start_time = time.time()
    
    # # Creates a Pandas series of s3 uris for this specific analysis layer
    # gross_emis_CO2_folder_interval = gross_emis_CO2_folder.replace("INTERVAL", interval)
    # gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder_interval)

    # gross_emis_all_gases_folder_interval = gross_emis_all_gases_folder.replace("INTERVAL", interval)
    # gross_emis_all_gases_uris = list_folder_uris(gross_emis_all_gases_folder_interval)
    
    # gross_remv_all_pools_folder_interval = gross_remv_all_pools_folder.replace("INTERVAL", interval)
    # gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder_interval)
    
    # net_flux_all_pools_CO2_folder_interval = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval)
    # net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder_interval)
    
    # # Creates a Pandas series of s3 uris for the relevant node codes
    # node_folder_interval = node_folder.replace("INTERVAL", interval)
    # node_tile_year_uris = list_folder_uris(node_folder_interval)

    gross_emis_CO2_uris = pd.Series([f's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/40000_pixels/20250511/{tile_id}__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif'])
    gross_emis_all_gases_uris = pd.Series([f's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/40000_pixels/20250511/{tile_id}__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif'])
    gross_remv_all_pools_uris = pd.Series([f's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/gross_removals__all_C_pools__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/40000_pixels/20250511/{tile_id}__gross_removals__all_C_pools__MgCO2_pixel_yr_{interval}.tif'])
    net_flux_all_pools_CO2_uris = pd.Series([f's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/net_flux__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/40000_pixels/20250511/{tile_id}__net_flux__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif'])
    node_uris = pd.Series([f's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_3/land_state_node/standard_model/annual_intervals/{interval}/40000_pixels/20250511/{tile_id}__land_state_node_{interval}.tif'])
   
    # print(gross_emis_CO2_folder)
    # print(gross_emis_CO2_uris[0])
    # print(gross_remv_all_pools_folder)
    # print(gross_remv_all_pools_uris[0])
    # print(net_flux_all_pools_CO2_folder)
    # print(net_flux_all_pools_CO2_uris[0])
    # print(node_folder)
    # print(node_tile_year_uris[0])
    # print(adm0_folder)
    # print(adm0_uris[0])
    
    # Gets input layer metadata, like the output pattern.
    # Note: chunk_id is for the first chunk being processed, not all chunks being processed.
    gross_emis_CO2_output_pattern = parse_metadata_from_uri(gross_emis_CO2_uris)
    gross_emis_all_gases_output_pattern = parse_metadata_from_uri(gross_emis_all_gases_uris)
    gross_remv_all_pools_output_pattern = parse_metadata_from_uri(gross_remv_all_pools_uris)
    net_flux_output_pattern = parse_metadata_from_uri(net_flux_all_pools_CO2_uris)
    node_output_pattern = parse_metadata_from_uri(node_uris)
    # print(gross_emis_CO2_output_pattern)
    # print(net_flux_output_pattern)
    
    print(f"   Reading gross emis CO2 only for {interval}: {timestr()}")
    gross_emis_CO2_xarray_chunks = make_xarray_chunks(gross_emis_CO2_uris, chunk_size)
    
    print(f"   Reading gross emis all gases for {interval}: {timestr()}")
    gross_emis_all_gases_xarray_chunks = make_xarray_chunks(gross_emis_all_gases_uris, chunk_size)
    
    print(f"   Reading gross removals for {interval}: {timestr()}")
    gross_remv_all_pools_xarray_chunks = make_xarray_chunks(gross_remv_all_pools_uris, chunk_size)
    
    print(f"   Reading net flux CO2 only for {interval}: {timestr()}")
    net_flux_all_pools_CO2_xarray_chunks = make_xarray_chunks(net_flux_all_pools_CO2_uris, chunk_size)
    
    print(f"   Reading state_nodes for {interval}: {timestr()}")
    node_xarray_chunks = make_xarray_chunks(node_uris, chunk_size)
  
    # print("gross_emis_CO2_xarray_chunks:", gross_emis_CO2_xarray_chunks)
    # print("gross_emis_all_gases_xarray_chunks:", gross_emis_all_gases_xarray_chunks)
    # print("gross_remv_all_pools_xarray_chunks:", gross_remv_all_pools_xarray_chunks)
    # print("net_flux_all_pools_CO2_xarray_chunks:", net_flux_all_pools_CO2_xarray_chunks)
    # print("node_xarray_chunks:", node_xarray_chunks)

    
    print(f"   Aligning {interval}: {timestr()}")
    gross_emis_CO2_aligned, nodes_aligned = align_with_nodes(gross_emis_CO2_xarray_chunks, node_xarray_chunks)
    gross_emis_all_gases_aligned, nodes_aligned = align_with_nodes(gross_emis_all_gases_xarray_chunks, node_xarray_chunks)
    gross_remv_all_pools_aligned, nodes_aligned = align_with_nodes(gross_remv_all_pools_xarray_chunks, node_xarray_chunks)
    net_flux_all_pools_CO2_aligned, nodes_aligned = align_with_nodes(net_flux_all_pools_CO2_xarray_chunks, node_xarray_chunks)

    adm0_aligned, nodes_aligned = align_with_nodes(adm0_xarray_chunks, node_xarray_chunks)
    pixel_area_aligned, nodes_aligned = align_with_nodes(pixel_area_xarray_chunks, node_xarray_chunks)
    
    # print("gross_emis_CO2_aligned:", gross_emis_CO2_aligned)
    # print("gross_emis_all_gases_aligned:", gross_emis_all_gases_aligned)
    # print("gross_remv_all_pools_aligned:", gross_remv_all_pools_aligned)
    # print("net_flux_all_pools_CO2_aligned:", net_flux_all_pools_CO2_aligned)
    # print("nodes_aligned:", nodes_aligned)
    # print("adm0_aligned:", adm0_aligned)
    # print("pixel_area_aligned:", pixel_area_aligned)
    
    nodes_aligned_data = nodes_aligned.band_data
    nodes_aligned_data.name = 'state_node'

    adm0_data = adm0_aligned.band_data
    adm0_data.name = 'gadm_adm0'

    print(f"   Stacking {interval}: {timestr()}")
    flux_cube = xr.DataArray(dask.array.stack((gross_emis_CO2_aligned.band_data, gross_emis_all_gases_aligned.band_data, 
                                               gross_remv_all_pools_aligned.band_data, net_flux_all_pools_CO2_aligned.band_data, pixel_area_aligned.band_data)), 
                             dims=('flux_type', 'y', 'x'))
    # flux_cube

    print(f"   Reducing {interval}: {timestr()}")
    # data_cube_by_node = xarray_reduction(flux_cube, nodes_aligned_data, adm0_data)

    # # Analysis for node_codes only
    # data_cube_by_contexts = xarray_reduce(
    #     flux_cube,
    #     nodes_aligned_data,
    #     func='sum',
    #     expected_groups=(node_codes),
    #     reindex=ReindexStrategy(
    #         blockwise=False, array_type=ReindexArrayType.SPARSE_COO
    #     ),
    #     fill_value=0
    # )

    # Analysis for node_codes and adm0
    data_cube_by_contexts = xarray_reduce(
        flux_cube,
        nodes_aligned_data,
        adm0_data,
        func='sum',
        expected_groups=(node_codes, gadm_adm0_ids),
        reindex=ReindexStrategy(
            blockwise=False, array_type=ReindexArrayType.SPARSE_COO
        ),
        fill_value=0
    )

    print(f"   Computing {interval}: {timestr()}")
    result = data_cube_by_contexts.compute()

    print(f"   Processing output for {interval}: {timestr()}")
    sparse_data = result.data

    dim_names = result.dims
    indices = sparse_data.coords
    values = sparse_data.data
    
    coord_dict = {
        dim: result.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    coord_dict["value"] = values

    # Creates the dataframe for the interval and does some processing of it
    df = create_interval_df(coord_dict)
    
    # df = pd.DataFrame(coord_dict)
    # df['flux_type'] = df['flux_type'].replace({0: gross_emis_CO2_output_pattern, 1: gross_emis_all_gases_output_pattern, 
    #                                            2: gross_remv_all_pools_output_pattern, 3: net_flux_output_pattern, 4: "area__ha"})
    # df['node_grp'] = df['state_node'].apply(classify_node)
    # df['state_node'] = 'n' + df['state_node'].astype(str)
    # df['interval_end'] = interval_end_year
    # # print(df)

    combined_df = pd.concat([combined_df, df])

    interval_end_time = time.time()
    print(f"   {interval} took {round(interval_end_time - interval_start_time)} seconds")

combined_df = combined_df.reset_index(drop=True)

analysis_end_time = time.time()
print(f"Analysis took {round(analysis_end_time - analysis_start_time)} seconds")

combined_df

Reading inputs that apply to all intervals: 20250518_21_04_44
   Reading adm0: 20250518_21_04_44
   Reading pixel_area: 20250518_21_04_46
Processing 2015_2016: 20250518_21_04_50
   Reading gross emis CO2 only for 2015_2016: 20250518_21_04_50
   Reading gross emis all gases for 2015_2016: 20250518_21_04_51
   Reading gross removals for 2015_2016: 20250518_21_04_53
   Reading net flux CO2 only for 2015_2016: 20250518_21_05_04
   Reading state_nodes for 2015_2016: 20250518_21_05_09
   Aligning 2015_2016: 20250518_21_05_19
   Stacking 2015_2016: 20250518_21_05_19
   Reducing 2015_2016: 20250518_21_05_19
   Computing 2015_2016: 20250518_21_05_19
   Processing output for 2015_2016: 20250518_21_06_28
   2015_2016 took 98 seconds
Processing 2016_2017: 20250518_21_06_28
   Reading gross emis CO2 only for 2016_2017: 20250518_21_06_28
   Reading gross emis all gases for 2016_2017: 20250518_21_06_28
   Reading gross removals for 2016_2017: 20250518_21_06_29
   Reading net flux CO2 only for 2016_20

,flux_type,state_node,gadm_adm0,value,node_grp,interval_end
0,gross_emissions__all_C_pools__CO2_only__MgCO2,n2111000,180,1.454755e+03,forest_loss,2016
1,gross_emissions__all_C_pools__CO2_only__MgCO2,n2112000,180,4.223097e+04,forest_loss,2016
2,gross_emissions__all_C_pools__CO2_only__MgCO2,n2112000,646,3.457110e+01,forest_loss,2016
3,gross_emissions__all_C_pools__CO2_only__MgCO2,n2112000,834,2.825665e+00,forest_loss,2016
4,gross_emissions__all_C_pools__CO2_only__MgCO2,n2121200,180,2.650796e+01,forest_loss,2016
...,...,...,...,...,...,...
5968,area__ha,n5320000,180,1.872893e+11,grassland,2023
5969,area__ha,n5320000,646,8.429135e+07,grassland,2023
5970,area__ha,n5320000,800,1.221080e+09,grassland,2023
5971,area__ha,n5320000,834,9.612528e+08,grassland,2023


In [61]:
# combined_df[(combined_df.flux_type == gross_remv_all_pools_output_pattern) 
# & (combined_df.gadm_adm0 == 180)]
combined_df[(combined_df.flux_type == 'area__ha') 
& (combined_df.state_node == 'n1110000')]

,flux_type,state_node,gadm_adm0,value,node_grp,interval_end
558,area__ha,n1110000,180,1.185682e+07,forest_gain,2016
559,area__ha,n1110000,834,7.662309e+02,forest_gain,2016
1292,area__ha,n1110000,180,1.151011e+07,forest_gain,2017
1293,area__ha,n1110000,646,4.611643e+03,forest_gain,2017
1294,area__ha,n1110000,834,3.831072e+03,forest_gain,2017
2033,area__ha,n1110000,180,4.058581e+06,forest_gain,2018
2034,area__ha,n1110000,646,9.223301e+03,forest_gain,2018
2035,area__ha,n1110000,834,6.129671e+03,forest_gain,2018
2789,area__ha,n1110000,180,1.445115e+06,forest_gain,2019
2790,area__ha,n1110000,646,1.076047e+04,forest_gain,2019


In [60]:
combined_df[(combined_df.flux_type == net_flux_output_pattern) 
& (combined_df.state_node == 'n1110000')]
# combined_df[(combined_df.state_node == 'n2112000')]
# combined_df.groupby(combined_df.gadm_adm0).sum()

,flux_type,state_node,gadm_adm0,value,node_grp,interval_end
367,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,180,-13146.842773,forest_gain,2016
368,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,834,-0.849597,forest_gain,2016
1112,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,180,-12762.407227,forest_gain,2017
1113,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,646,-5.113389,forest_gain,2017
1114,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,834,-4.247893,forest_gain,2017
1847,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,180,-4500.155273,forest_gain,2018
1848,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,646,-10.226795,forest_gain,2018
1849,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,834,-6.796580,forest_gain,2018
2600,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,180,-1602.343018,forest_gain,2019
2601,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,646,-11.931206,forest_gain,2019


In [64]:
combined_df[(combined_df.state_node == 'n1110000')]

,flux_type,state_node,gadm_adm0,value,node_grp,interval_end
298,gross_removals__all_C_pools__MgCO2,n1110000,180,-1.314684e+04,forest_gain,2016
299,gross_removals__all_C_pools__MgCO2,n1110000,834,-8.495969e-01,forest_gain,2016
367,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,180,-1.314684e+04,forest_gain,2016
368,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,834,-8.495969e-01,forest_gain,2016
558,area__ha,n1110000,180,1.185682e+07,forest_gain,2016
559,area__ha,n1110000,834,7.662309e+02,forest_gain,2016
1043,gross_removals__all_C_pools__MgCO2,n1110000,180,-1.276241e+04,forest_gain,2017
1044,gross_removals__all_C_pools__MgCO2,n1110000,646,-5.113389e+00,forest_gain,2017
1045,gross_removals__all_C_pools__MgCO2,n1110000,834,-4.247893e+00,forest_gain,2017
1112,net_flux__all_C_pools__CO2_only__MgCO2,n1110000,180,-1.276241e+04,forest_gain,2017


In [65]:
# Calculation of flux densities (Mg/ha) per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682a8c76-0618-800a-a201-18fd404a281f
# Step 1: Filter out area and flux data
area_df = combined_df[combined_df['flux_type'] == 'area__ha'].copy()
flux_df = combined_df[combined_df['flux_type'] != 'area__ha'].copy()

In [76]:
# Step 2: Merge flux data with area data on matching keys
merged = pd.merge(
    flux_df,
    area_df[['state_node', 'gadm_adm0', 'interval_end', 'value']],
    on=['state_node', 'gadm_adm0', 'interval_end'],
    how='left',
    suffixes=('', '_area')
)
merged

,flux_type,state_node,gadm_adm0,value,node_grp,interval_end,value_area
0,gross_emissions__all_C_pools__CO2_only__MgCO2,n2111000,180,1.454755e+03,forest_loss,2016,4.680771e+04
1,gross_emissions__all_C_pools__CO2_only__MgCO2,n2112000,180,4.223097e+04,forest_loss,2016,9.180095e+05
2,gross_emissions__all_C_pools__CO2_only__MgCO2,n2112000,646,3.457110e+01,forest_loss,2016,2.305799e+03
3,gross_emissions__all_C_pools__CO2_only__MgCO2,n2112000,834,2.825665e+00,forest_loss,2016,7.662283e+02
4,gross_emissions__all_C_pools__CO2_only__MgCO2,n2121200,180,2.650796e+01,forest_loss,2016,1.537415e+03
...,...,...,...,...,...,...,...
4321,net_flux__all_C_pools__CO2_only__MgCO2,n5220000,180,1.870412e+06,grassland,2023,5.400559e+08
4322,net_flux__all_C_pools__CO2_only__MgCO2,n5220000,646,6.680202e+03,grassland,2023,9.661300e+05
4323,net_flux__all_C_pools__CO2_only__MgCO2,n5220000,800,2.341988e+04,grassland,2023,5.787288e+06
4324,net_flux__all_C_pools__CO2_only__MgCO2,n5220000,834,1.559471e+04,grassland,2023,3.944130e+06


In [77]:
# Step 3: Compute per-hectare flux
merged['value_per_ha'] = merged['value'] / merged['value_area'] * 10000
merged

,flux_type,state_node,gadm_adm0,value,node_grp,interval_end,value_area,value_per_ha
0,gross_emissions__all_C_pools__CO2_only__MgCO2,n2111000,180,1.454755e+03,forest_loss,2016,4.680771e+04,310.793823
1,gross_emissions__all_C_pools__CO2_only__MgCO2,n2112000,180,4.223097e+04,forest_loss,2016,9.180095e+05,460.027618
2,gross_emissions__all_C_pools__CO2_only__MgCO2,n2112000,646,3.457110e+01,forest_loss,2016,2.305799e+03,149.931137
3,gross_emissions__all_C_pools__CO2_only__MgCO2,n2112000,834,2.825665e+00,forest_loss,2016,7.662283e+02,36.877586
4,gross_emissions__all_C_pools__CO2_only__MgCO2,n2121200,180,2.650796e+01,forest_loss,2016,1.537415e+03,172.419052
...,...,...,...,...,...,...,...,...
4321,net_flux__all_C_pools__CO2_only__MgCO2,n5220000,180,1.870412e+06,grassland,2023,5.400559e+08,34.633671
4322,net_flux__all_C_pools__CO2_only__MgCO2,n5220000,646,6.680202e+03,grassland,2023,9.661300e+05,69.143929
4323,net_flux__all_C_pools__CO2_only__MgCO2,n5220000,800,2.341988e+04,grassland,2023,5.787288e+06,40.467800
4324,net_flux__all_C_pools__CO2_only__MgCO2,n5220000,834,1.559471e+04,grassland,2023,3.944130e+06,39.539036


In [78]:
# Step 4: Prepare new rows to append
new_rows = merged.copy()
new_rows['flux_type'] = new_rows['flux_type'] + '__per_ha'
new_rows['value'] = new_rows['value_per_ha']
new_rows = new_rows.drop(columns=['value_area', 'value_per_ha'])
new_rows

,flux_type,state_node,gadm_adm0,value,node_grp,interval_end
0,gross_emissions__all_C_pools__CO2_only__MgCO2_...,n2111000,180,310.793823,forest_loss,2016
1,gross_emissions__all_C_pools__CO2_only__MgCO2_...,n2112000,180,460.027618,forest_loss,2016
2,gross_emissions__all_C_pools__CO2_only__MgCO2_...,n2112000,646,149.931137,forest_loss,2016
3,gross_emissions__all_C_pools__CO2_only__MgCO2_...,n2112000,834,36.877586,forest_loss,2016
4,gross_emissions__all_C_pools__CO2_only__MgCO2_...,n2121200,180,172.419052,forest_loss,2016
...,...,...,...,...,...,...
4321,net_flux__all_C_pools__CO2_only__MgCO2__per_ha,n5220000,180,34.633671,grassland,2023
4322,net_flux__all_C_pools__CO2_only__MgCO2__per_ha,n5220000,646,69.143929,grassland,2023
4323,net_flux__all_C_pools__CO2_only__MgCO2__per_ha,n5220000,800,40.467800,grassland,2023
4324,net_flux__all_C_pools__CO2_only__MgCO2__per_ha,n5220000,834,39.539036,grassland,2023


In [79]:
# Step 5: Append to original dataframe
result_df = pd.concat([df, new_rows], ignore_index=True)
result_df

,flux_type,state_node,gadm_adm0,value,node_grp,interval_end
0,gross_emissions__all_C_pools__CO2_only__MgCO2,n2111000,180,107.011795,forest_loss,2023
1,gross_emissions__all_C_pools__CO2_only__MgCO2,n2112000,180,55683.566406,forest_loss,2023
2,gross_emissions__all_C_pools__CO2_only__MgCO2,n2112000,834,10.043992,forest_loss,2023
3,gross_emissions__all_C_pools__CO2_only__MgCO2,n2121200,180,36.964756,forest_loss,2023
4,gross_emissions__all_C_pools__CO2_only__MgCO2,n2121200,646,227.305740,forest_loss,2023
...,...,...,...,...,...,...
5059,net_flux__all_C_pools__CO2_only__MgCO2__per_ha,n5220000,180,34.633671,grassland,2023
5060,net_flux__all_C_pools__CO2_only__MgCO2__per_ha,n5220000,646,69.143929,grassland,2023
5061,net_flux__all_C_pools__CO2_only__MgCO2__per_ha,n5220000,800,40.467800,grassland,2023
5062,net_flux__all_C_pools__CO2_only__MgCO2__per_ha,n5220000,834,39.539036,grassland,2023


In [111]:
result_df[(result_df.flux_type == f'{gross_remv_all_pools_output_pattern}__per_ha') 
& (result_df.state_node == 'n5100000')]

,flux_type,state_node,gadm_adm0,value,node_grp,interval_end
1099,gross_removals__all_C_pools__MgCO2__per_ha,n5100000,24,-21.888813,grassland,2016
1100,gross_removals__all_C_pools__MgCO2__per_ha,n5100000,180,-21.866413,grassland,2016
1101,gross_removals__all_C_pools__MgCO2__per_ha,n5100000,646,-22.388397,grassland,2016
1102,gross_removals__all_C_pools__MgCO2__per_ha,n5100000,800,-22.689709,grassland,2016
1103,gross_removals__all_C_pools__MgCO2__per_ha,n5100000,834,-22.424379,grassland,2016
1104,gross_removals__all_C_pools__MgCO2__per_ha,n5100000,894,-21.908104,grassland,2016
1632,gross_removals__all_C_pools__MgCO2__per_ha,n5100000,24,-21.848877,grassland,2017
1633,gross_removals__all_C_pools__MgCO2__per_ha,n5100000,180,-21.973492,grassland,2017
1634,gross_removals__all_C_pools__MgCO2__per_ha,n5100000,646,-22.315031,grassland,2017
1635,gross_removals__all_C_pools__MgCO2__per_ha,n5100000,800,-22.836143,grassland,2017


In [81]:
walker = pyg.walk(result_df)

Box(children=(HTML(value='<div id="ifr-pyg-4" style="height: auto">\n    <head>\n        <meta http-equiv="Con…